In [2]:
import json
import re
import pandas as pd

In [3]:
def nettoyer_texte_llm(texte):
    """
    Nettoie les réponses des LLMs en supprimant les introductions
    et les listes de modifications à la fin.
    """
    if not isinstance(texte, str) or not texte:
        return ""

    texte = texte.strip()
    # Supprimer l'introduction du LLM (préfixe)
    pattern_intro = (
        r"^(?:here(?:'s| is)|sure|certainly|below|this is|as requested|"
        r"of course|absolutely|happy to).*?(?::|\.)\s*\n+"
    )
    texte = re.sub(pattern_intro, "", texte, flags=re.IGNORECASE)

    # Supprimer la conclusion / liste de changements (suffixe)
    pattern_outro = (
        r"(\n\s*(?:i made the following changes|changes made|"
        r"here are the changes|notes?):).*$"
    )
    texte = re.sub(pattern_outro, "", texte, flags=re.IGNORECASE | re.DOTALL)

    return texte.strip()

In [10]:
CHEMIN_JSON = "../data/json/rewrites.json"
colonne_originale = "Paragraphes"
colonnes_generees = ["Rew1", "Rew2", "Rew3"]

try:
    with open(CHEMIN_JSON, "r", encoding='utf-8') as file:
        df_rewrites = pd.DataFrame(json.load(file))

    for col in colonnes_generees:
        if col in df_rewrites.columns:
            df_rewrites[col] = df_rewrites[col].apply(nettoyer_texte_llm)

    print("Données chargées et nettoyées avec succès !")

    cols_a_analyser = [col for col in [colonne_originale] + colonnes_generees if col in df_rewrites.columns]

    for col in cols_a_analyser:
        nom_col_longueur = f"Mots_{col}"
        df_rewrites[nom_col_longueur] = df_rewrites[col].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

    cols_longueurs = [f"Mots_{col}" for col in cols_a_analyser]

    print("Statistiques descriptives du corpus (Nombre de mots)")

    # 1. Création du tableau de statistiques
    stats_mots = df_rewrites[cols_longueurs].describe().T[['count', 'mean', 'std', '50%', 'min', 'max']].round(1)

    # Renommage propre des colonnes
    stats_mots.columns = ['Nb de textes', 'Moyenne', 'Écart-type', 'Médiane', 'Minimum', 'Maximum']

    # On s'assure que le 'Nb de textes' soit un entier
    stats_mots['Nb de textes'] = stats_mots['Nb de textes'].astype(int)

    display(stats_mots)

    print(stats_mots)

except FileNotFoundError:
    print(f"Fichier introuvable : {CHEMIN_JSON}")

Données chargées et nettoyées avec succès !
Statistiques descriptives du corpus (Nombre de mots)


,Nb de textes,Moyenne,Écart-type,Médiane,Minimum,Maximum
Mots_Paragraphes,121,189.9,81.7,176.0,67.0,535.0
Mots_Rew1,121,156.0,37.8,158.0,75.0,226.0
Mots_Rew2,121,163.0,41.6,173.0,63.0,234.0
Mots_Rew3,121,161.9,41.3,168.0,66.0,234.0


                  Nb de textes  Moyenne  Écart-type  Médiane  Minimum  Maximum
Mots_Paragraphes           121    189.9        81.7    176.0     67.0    535.0
Mots_Rew1                  121    156.0        37.8    158.0     75.0    226.0
Mots_Rew2                  121    163.0        41.6    173.0     63.0    234.0
Mots_Rew3                  121    161.9        41.3    168.0     66.0    234.0
